In [ ]:
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from datasets import load_from_disk

In [ ]:
checkpoint_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_id)

In [ ]:
tokenized_datasets = load_from_disk("./dataset/tokenized_datasets")

In [ ]:
tokenized_datasets

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
  f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]

  return {"accuracy": acc, "f1": f1}


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_id, num_labels=3)

In [ ]:
training_args = TrainingArguments(
    output_dir="bert-twitter-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}

model.config.id2label = id2label
model.config.label2id = label2id
local_model_path = "./bert-twitter-sentiment"


In [ ]:
trainer.save_model(local_model_path)
classifier = pipeline(
    "text-classification",
    model=local_model_path,
    device_map="auto"
)

In [ ]:
test_tweets = [
    "I absolutely love the new features on this app, amazing job!",
    "This update completely ruined my day. Worst interface ever."
]

results = classifier(test_tweets)

for tweet, result in zip(test_tweets, results):
    print(f"Tweet: '{tweet}'\nPrediction: {result['label']} | Confidence: {result['score']:.2%}\n")